# Solutions · Chapter 02-03 · The rows you never see

Worked answers with reasoning. E4 and E14 are the two that change how you think: one shows that
tripling your response rate can leave the bias *exactly* unchanged, the other shows reweighting
working and then failing for the reason that matters.

Self-contained: run from the top with a fresh kernel.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(21)
n_customers = 2000
satisfaction = np.clip(rng.normal(3.4, 1.0, n_customers), 1, 5).round(1)
response_chance = 0.03 + 0.40 * (satisfaction - 1) / 4
responded = np.random.default_rng(7).random(n_customers) < response_chance
customers = pd.DataFrame({"satisfaction": satisfaction, "responded": responded})
print(f"{n_customers} customers, {responded.sum()} responses "
      f"({responded.mean():.1%}), true mean {satisfaction.mean():.2f}")

## E1 · Population, frame, sample

- **Population** - everyone the conclusion is meant to be about. Maria's 2,000 customers.
- **Sampling frame** - everyone who *could* have ended up in the data. Customers who came to the
  counter during the month.
- **Sample** - everyone who actually appears. The 549 who filled in a card.

**The gaps:**

- **Population to frame: coverage.** App-only customers had zero chance of being surveyed. No amount
  of extra collection reaches them, because the method cannot see them.
- **Frame to sample: non-response.** Of those who could have answered, most did not.

The distinction matters because the fixes are different. Coverage is fixed by changing *where* you
collect; non-response by changing *who answers* or by measuring the people who did not.

## E2 · Why more cards do not help

Doubling the cards reduces **variance** - the random wobble around whatever number the process
converges to. It does nothing to **bias** - the gap between that number and the truth.

The survey estimates a well-defined quantity: *the mean satisfaction among customers who choose to
respond*. More data estimates that quantity more precisely. It is simply not the quantity anyone
wanted.

**The mental image worth keeping:** a bathroom scale that reads 3 kg heavy. Standing on it a
thousand times gives you a very precise estimate - of the wrong weight. Repetition addresses noise;
only recalibration addresses offset, and recalibration means finding out about the people who did
not answer.

**And there is a trap in the improvement:** as the sample grows, the confidence interval narrows, so
the wrong answer arrives with *increasing* apparent authority. A biased estimate looks most credible
exactly when it is worst supported.

## E3 · Why `.isna()` cannot see it

`.isna()` inspects rows that **exist**. Selection bias is about rows that were **never created**.

A customer who did not fill in a card does not appear as a row of `NaN`s - they appear not at all.
The DataFrame's row count is 549, its shape is consistent, and every value in it is real. There is
nothing in the object to count.

**This is the structural reason 02-03 and 02-04 are separate chapters.** Missing *values* are a
data-quality problem: countable, visible, fixable with a stated policy. Missing *rows* are a
sampling problem: invisible in the file, and answerable only by knowing how the file was built. The
first is solved with code, the second with provenance.

**The one trace selection sometimes leaves:** a row count that does not match a total you know from
elsewhere. 549 cards against 2,000 customers is the whole finding, and it is available only because
someone knew the customer count. **Always compare your row count with an external total.**

## E4 · Tripling the response rate

In [ ]:
def survey_arithmetic(p_satisfied, p_dissatisfied):
    n_sat, n_dis, rating_sat, rating_dis = 700, 300, 4, 2
    r_sat, r_dis = n_sat * p_satisfied, n_dis * p_dissatisfied
    survey_mean = (r_sat * rating_sat + r_dis * rating_dis) / (r_sat + r_dis)
    true_mean = (n_sat * rating_sat + n_dis * rating_dis) / (n_sat + n_dis)
    print(f"p={p_satisfied:.2f}/{p_dissatisfied:.2f} -> responses {r_sat:.0f} satisfied, "
          f"{r_dis:.0f} dissatisfied ({r_sat + r_dis:.0f} total) | survey {survey_mean:.2f}, "
          f"true {true_mean:.2f}, bias {survey_mean - true_mean:+.2f}")

survey_arithmetic(0.30, 0.10)
survey_arithmetic(0.60, 0.20)

**First case:** 210 satisfied and 30 dissatisfied respond, 240 cards. Survey mean
`(210x4 + 30x2) / 240 = 900 / 240 = 3.75`. True mean `3.40`. **Bias +0.35.**

**Second case:** 420 and 60, so 480 cards - *twice as many*. Survey mean
`(420x4 + 60x2) / 480 = 1800 / 480 = 3.75`. **Bias +0.35, exactly unchanged.**

**Why.** The survey mean depends on the *composition* of respondents, and composition depends on the
**ratio** of the two response probabilities, not their level. In both cases satisfied customers are
three times as likely to respond, so respondents are 87.5% satisfied against a true 70% - identical
in both.

Tripling both rates triples the number of cards and leaves the skew untouched.

**The practical consequence is the useful part.** "We must improve our response rate" is the standard
answer to survey doubt, and by itself it is worthless. What matters is whether the improvement is
*differential* - whether it brings in the people who were not answering. A campaign that gets more
responses from everyone equally changes nothing at all.

**The generalisation:** whenever a fix increases your data uniformly, it addresses variance. Only a
fix that changes *who* is included addresses bias.

## E5 · How much could the censoring hide?

In [ ]:
n_total, share_running = 3000, 0.162
n_running = round(n_total * share_running)
n_done = n_total - n_running
observed_mean = 16.0

for assumed in (30, 60, 120):
    implied = (n_done * observed_mean + n_running * assumed) / n_total
    print(f"if every unfinished rental lasted {assumed:3d} min -> true mean {implied:.1f} min "
          f"({implied / observed_mean - 1:+.0%} vs observed)")

With 486 rentals still running and 2,514 completed at 16 minutes: if each unfinished rental turned
out to last exactly 60 minutes, the true mean would be **23.1 minutes** - 44% above the observed 16.

**What that tells you.** The observed mean is a *lower bound*, and the size of the gap depends
entirely on a quantity you did not observe. Assume 30 minutes and the true mean is 18.3; assume 120
and it is 32.8. Three defensible assumptions, three very different answers, and the data cannot
choose between them.

**Why this is the right way to reason about it.** When you cannot estimate something, bound it.
Running the calculation across a range of plausible assumptions - a sensitivity analysis - converts
"we do not know" into "the answer is between 18 and 33, and here is what it depends on". That is a
usable statement, and it makes the assumption visible instead of hidden.

**One thing you *do* know for free:** every censored rental has already lasted at least
`snapshot - start_time`. Those minutes are observed. Throwing them away by dropping the row discards
real information, and using them is exactly what survival analysis does (12-07).

## E6 · A response-bias report

In [ ]:
def response_bias(frame, value_column, responded_column):
    values, mask = frame[value_column], frame[responded_column]
    print(f"response rate      : {mask.mean():.1%}  ({mask.sum()} of {len(frame)})")
    print(f"population mean    : {values.mean():.3f}")
    print(f"respondent mean    : {values[mask].mean():.3f}")
    print(f"bias               : {values[mask].mean() - values.mean():+.3f}")
    quartile = pd.qcut(values, 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"], duplicates="drop")
    print("\nresponse rate within each quartile of", value_column)
    print(mask.groupby(quartile, observed=True).mean().mul(100).round(1).to_string())


response_bias(customers, "satisfaction", "responded")

The quartile breakdown is the diagnostic that matters, and it is the one that generalises.

A flat response rate across quartiles would mean non-response is unrelated to the value - annoying
but unbiased. A rate that climbs from 15.7% in the lowest quartile to 41.4% in the highest says it as loudly as data
can.

**And here is the catch, which is the whole difficulty of the subject:** this report can only be
produced because we generated the population and know everyone's satisfaction. In real work the last
block is impossible - you do not know the satisfaction of people who did not tell you.

**What you can do instead** is run exactly this breakdown on fields you *do* know for everyone:
tenure, plan, region, how often they visit. If the response rate varies strongly by tenure, the
respondents are not representative on tenure, and there is no reason to believe they are
representative on satisfaction either. It does not measure the bias - it establishes that one
exists, which is usually enough to change how the number is reported.

## E7 · Fifty fair responses against a thousand biased ones

In [ ]:
truth = satisfaction.mean()
fair_means, biased_means = [], []
gen = np.random.default_rng(5)

for _ in range(500):
    fair_idx = gen.choice(n_customers, 50, replace=False)                 # a proper random sample
    fair_means.append(satisfaction[fair_idx].mean())

    chance = np.minimum(response_chance * 3.0, 1.0)                       # a big, uniform push
    got = gen.random(n_customers) < chance
    biased_means.append(satisfaction[got].mean())

fair_means, biased_means = np.array(fair_means), np.array(biased_means)
print(f"truth                     : {truth:.3f}")
print(f"fair sample of ~50        : mean {fair_means.mean():.3f}, sd {fair_means.std():.3f}, "
      f"error {abs(fair_means.mean() - truth):.3f}")
print(f"biased sample of ~{int((np.minimum(response_chance * 3, 1)).sum())}      : "
      f"mean {biased_means.mean():.3f}, sd {biased_means.std():.3f}, "
      f"error {abs(biased_means.mean() - truth):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.hist(fair_means, bins=30, alpha=0.75, color="#0072B2", label="fair sample of 50")
ax.hist(biased_means, bins=30, alpha=0.85, color="#D55E00", label="biased sample of ~1,300")
ax.axvline(truth, color="#009E73", linewidth=2.5, label=f"truth = {truth:.2f}")
ax.set_xlabel("Estimated mean satisfaction"); ax.set_ylabel("Number of repetitions")
ax.set_title("A small fair sample beats a large biased one")
ax.legend(fontsize=8)
plt.show()

**The fifty fair responses win, and it is not close.**

The fair sample is *wide* - a standard deviation of 0.12 across repetitions - but it is
centred on the truth, landing on 3.417 against a true 3.415.

The biased sample of roughly 1,550 is *narrow* - a standard deviation of 0.01, twelve times tighter - and
that answer is 0.256 too high. Its precision is real, and it is precision about the wrong
quantity.

**Which would I rather have?** The fifty, without hesitation. A wide estimate around the right value
can be narrowed by collecting more of the same kind. A narrow estimate around the wrong value cannot
be fixed by anything except changing how it is collected.

**The historical case is worth knowing**, because it is the canonical example and the numbers are
absurd. In 1936 the *Literary Digest* mailed ten million ballots and received 2.4 million responses -
one of the largest polls ever conducted - and predicted the wrong winner of the US presidential
election by a wide margin. George Gallup surveyed a few thousand people, chosen to represent the
population, and got it right. The *Digest* sampled from telephone directories and car registrations
in the middle of the Depression, and being able to afford a telephone was related to how people
voted. Two and a half million responses could not repair a frame that excluded the poor.

## E8 · "Our app store rating is 4.6"

**Three selection mechanisms that produce a 4.6 from a mediocre product:**

1. **Self-selection at the extremes, plus prompting.** Most users never rate anything. Those who do
   are the delighted and the furious - and many apps ask for a rating *only after a successful
   action*, or show the store prompt only to users who have opened the app ten times. That prompt
   design is a selection mechanism deliberately engineered to produce 4.6.
2. **Survivorship.** Only current users can rate. Everyone who tried the app, disliked it and
   uninstalled is gone from the frame. The rating describes people who stayed, which is the group
   least likely to be dissatisfied.
3. **Version and recency effects.** Store ratings are often shown for the current version only, so a
   bad release can be reset away. And ratings accumulate from early enthusiasts - the people who seek
   out a new app are not the people who arrive later through an ad.

**The one piece of data I would ask for: the rating distribution alongside retention.** A 4.6 built
from 80% fives and 12% ones is a bimodal product with a segment it is failing, not a product
everyone likes. Cross that with what share of installers are still active at 30 days and you learn
in one table what the average conceals.

**The line worth carrying:** an average rating measures *the people who rated*, and app stores are
built to make that group unrepresentative on purpose.

## E9 · The hiring model

**The selection mechanism: selection on the outcome, through a filter you did not build.** The
training data contains performance reviews only for people who were **hired**. Applicants who were
rejected have no outcome, because nobody ever observed how they would have performed. The dataset is
not a sample of applicants; it is a sample of applicants *who passed the existing screen*.

Three consequences, all of them invisible in the file:

- The model learns what predicts success **among people the old process already liked**. Anyone the
  old process filtered out is outside the training distribution entirely, and the model extrapolates
  into that region with no evidence.
- If the old screen was good on some dimension, that dimension shows little variation among the
  hired, so the model concludes it does not matter - and stops screening for it. This is the most
  counter-intuitive part: **a criterion that works well disappears from the data precisely because it
  was applied.**
- Any bias in the old screen is inherited and laundered into a model, which then looks objective.
  That is 13-01's subject.

**Why cross-validation could not catch it.** Cross-validation splits the *training data* - all of
whom were hired - so every fold is drawn from the same filtered population. It measures how well the
model predicts performance among hired people, which it does honestly. The failure appears only
against applicants the filter would have rejected, and there are none in any fold. **Cross-validation
verifies internal consistency, never external validity.**

**What would fix it properly.** Outcomes for people the current process rejects - which means
deliberately hiring some applicants the screen would have declined, or using a period or location
where the screen was not applied, or an existing randomised element in the process. Every one of
these is expensive and organisationally hard, and there is no analytical substitute. Where none is
possible, the honest position is that the model can rank *within* the screened pool and cannot
evaluate the screen itself.

## E10 · "We have every transaction, so it isn't a sample"

> You have every transaction *that happened*, which is not the same as every transaction that could
> have happened - and the ones that did not happen are usually the question. Customers who left
> before buying, orders abandoned at checkout, people the pricing turned away and applicants the
> system rejected are all absent, and their absence is systematic rather than random. A complete
> census of your own system is still a sample of the market, selected by everything your system does:
> who it reaches, who it approves, who it prices out. So the risk is not that the rows are unrepresentative
> of your transactions - they are your transactions - it is that a conclusion phrased about
> *customers*, *demand* or *the market* is being drawn from data about *transactions you completed*.

**What is being tested:** whether you can hold the distinction between the population you *have* and
the population your *conclusion* is about. "Population data" is a claim about the first; selection
bias is a property of the relationship between the two.

## E11 · Designing a survey Maria can trust

**Where and when.** Not a card on the counter. Sample from the *rental records*, which cover every
customer, and contact a randomly chosen subset - by app notification and by email, with a paper
option at the rack for people who use neither. That fixes coverage: everyone has a known, non-zero
chance of selection.

**Whom.** A random 300, not "whoever comes by". Stratify by usage band - light, medium, heavy - and
sample within each, so the light users who are easiest to lose are not swamped by the regulars who
are easiest to reach.

**How to raise response without skewing it.** A short survey (three questions), a small reward
offered to *everyone* selected regardless of whether they answer, and up to two reminders. The
reward must not depend on responding, or you have selected on willingness again - the classic
version of this mistake.

**How to check afterwards that it worked.** Compare respondents with the full selected sample on
everything already known from the records - usage frequency, tenure, plan, station, recency. If they
match on all of those, representativeness is plausible. If heavy users are twice as likely to have
answered, they are not, and the number needs reweighting or a caveat.

**The strongest check, if the budget allows:** chase a random 30 of the non-responders by phone. That
converts the unknown into a measurement, and it is the only method here that does.

**What the design still cannot fix:** people who refuse *because* of how they feel. If the most
dissatisfied customers systematically ignore every contact - having already decided the company is
not worth their time - no amount of design reaches them, and the estimate stays optimistic by an
unknown amount. Every survey has this residual. **The honest response is to state it, not to solve
it.**

## E12 · Five mechanisms in the wild

| | Mechanism | Direction of the bias |
|---|---|---|
| (a) Customer lifetime from customers who cancelled | **Censoring** (plus survivorship) | **Understates** lifetime - the loyal customers have not cancelled yet, so only the short-lived ones are measured |
| (b) Treatment effect from patients who completed the course | **Selection on the outcome** | **Overstates** effectiveness - people drop out because it is not working or has side effects |
| (c) Graduate salary from an alumni survey | **Self-selection** | **Overstates** salary - success makes people more willing to answer, and easier to contact |
| (d) Fraud rate from flagged and reviewed transactions | **Selection on the outcome** | Unpredictable in size, and structurally blind - you only ever confirm fraud the existing system already suspects |
| (e) Code-review duration from finished reviews | **Censoring** | **Understates** duration - the reviews still open are the slow ones |

**(a) and (e) are the same shape** and it is the most common one in industry: any "average time to X"
computed from the cases that reached X. The rule from the chapter applies without exception - it is
always too short.

**(d) is the worst of the five**, because unlike the others it does not merely shift a number, it
defines what you can learn. A model trained on confirmed fraud learns the *current detector's*
notion of fraud. Novel fraud is not under-represented; it is absent, and no amount of data collected
the same way will contain it. That is a feedback loop (11-07) as well as a selection problem.

## E13 · Explaining it to Maria

> The cards only tell you about the people who filled one in. Someone who had a good time is
> standing at the counter feeling friendly and happy to scribble a five. Someone who waited twenty
> minutes for a broken bike has already walked off. So the pile on your desk is missing exactly the
> people you most want to hear from - which is why 3.7 is probably kinder than the truth.

(66 words.)

**The move that makes this explanation work:** it describes two *specific people* and what each of
them does with a card, rather than describing a statistical property. Maria can picture both, and
once she has pictured the second one she has understood the entire chapter.

## E14 · Reweighting: when it works and when it does not

In [ ]:
gen = np.random.default_rng(31)
n = 5000

plan = gen.choice(["basic", "premium"], n, p=[0.7, 0.3])
value = np.clip(gen.normal(3.4, 1.0, n) + (plan == "premium") * 0.5, 1, 5)   # premium are happier

# World A: responding depends on PLAN, which we record for everyone.
p_a = np.where(plan == "premium", 0.50, 0.15)
resp_a = gen.random(n) < p_a

# World B: responding depends on SATISFACTION itself, which we never observe for non-responders.
p_b = 0.03 + 0.40 * (value - 1) / 4
resp_b = gen.random(n) < p_b

def reweighted_mean(values, responded, groups):
    """Inverse-probability weighting using the response rate within each observed group."""
    rate = pd.Series(responded).groupby(groups).transform("mean").to_numpy()
    w = 1 / rate[responded]
    return float(np.average(values[responded], weights=w))

print(f"truth                                : {value.mean():.3f}\n")
for label, resp in [("A: response depends on PLAN", resp_a), ("B: response depends on SATISFACTION", resp_b)]:
    naive = value[resp].mean()
    fixed = reweighted_mean(value, resp, plan)
    print(f"{label}\n   naive mean {naive:.3f} ({naive - value.mean():+.3f})"
          f" | reweighted by plan {fixed:.3f} ({fixed - value.mean():+.3f})")

**World A: reweighting works almost perfectly.** Premium customers answer more often and are
happier, so the raw mean is too high - but because the *reason* for responding is `plan`, and `plan`
is recorded for everyone, dividing by each group's response rate restores the population
composition. The corrected estimate cuts the error from +0.151 to +0.027.

**World B: reweighting barely helps.** Responding depends on satisfaction itself. Within any plan,
the people who answered are still the happier ones, and the weights - which only know about plan -
cannot see that. The correction moves the number from +0.317 to +0.309 - a rounding error - because plan and satisfaction are
only weakly related, and it leaves essentially all of the bias in place.

**The rule this establishes, and it is the whole point:**

> **Reweighting corrects for selection on things you measured. It cannot correct for selection on
> things you did not.**

The uncomfortable part is that **the two worlds look identical in the data you hold.** In both, you
see plan, you see responses, you see a response rate that varies by plan, and you can compute a
weighted mean. Nothing distinguishes them. Deciding which world you are in is not a calculation; it
is a judgement about *why* people respond, and it comes from knowing the process - which is 02-02.

**What a careful analyst does:** reweight anyway, since it can only help, then report both numbers
and state plainly that the correction handles the observable differences and cannot address whether
non-responders differ in ways not recorded. That sentence is the deliverable. This is the same
honesty required of "we controlled for it" in 00-04's E7, and it is the same underlying limitation:
adjustment reaches exactly as far as your measurements do.

---

## Where to go next

Back to the chapter for the mastery check and flashcards, then **02-04 · Missing values, duplicates,
impossible values and messy categories** - the defects that do leave a trace.